# Colab: Experiment 2 Production Runs

This notebook runs the finalized Experiment 2 setup on Colab GPU.

- `completion` = Experiment 2a (constrained completion-choice)
- `generation` = Experiment 2b (looser generation)
- `both` = run both sequentially

Recommended main baselines:

- `active`
- `passive`
- `no_prime_eos`
- `filler`

`no_prime_empty` can be added later as a robustness control, but it is not included in the default Colab run.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path('/content/Geometry-of-Syntax')
REPO_URL = 'https://github.com/<YOUR-USERNAME>/Geometry-of-Syntax.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/Geometry-of-Syntax
!git pull --ff-only || true
!nvidia-smi || true
!python --version

# Keep Colab's CUDA torch; only install the Python-side deps we need.
!pip install -q pandas scipy statsmodels tqdm "transformers>=4.57,<5"

# Colab compatibility: older repo copies may not include the local PrimeLM verb list.
common_path = REPO_DIR / 'scripts' / 'production_priming_common.py'
common_text = common_path.read_text()
needle = "def load_verb_lookup(path: Path = VERB_LIST_PATH) -> Dict[Tuple[str, str], str]:\n    frame = pd.read_csv(path, sep=\";\")\n"
replacement = "def load_verb_lookup(path: Path = VERB_LIST_PATH) -> Dict[Tuple[str, str], str]:\n    if not path.exists():\n        warnings.warn(\n            f\"Verb lookup file not found at {path}; falling back to heuristic lemma inference.\",\n        )\n        return {}\n    frame = pd.read_csv(path, sep=\";\")\n"
if needle in common_text and 'falling back to heuristic lemma inference' not in common_text:
    common_path.write_text(common_text.replace(needle, replacement))
    print('Applied Colab compatibility patch for missing PrimeLM verb list.')


In [ ]:
MODEL_NAME = 'gpt2-large'
WHICH = 'both'  # 'completion', 'generation', or 'both'
BATCH_SIZE = 64
MAX_ITEMS = 2080
PROMPT_TEMPLATE = 'role_labeled'
ROLE_ORDER = 'shuffle'
PRIME_CONDITIONS = ['active', 'passive', 'no_prime_eos', 'filler']
PRIME_CONDITIONS_ARG = ' '.join(PRIME_CONDITIONS)
MAX_NEW_TOKENS = 24
SEED = 13
OUTPUT_ROOT = 'behavioral_results/experiment_2_colab'

print({
    'model_name': MODEL_NAME,
    'which': WHICH,
    'batch_size': BATCH_SIZE,
    'max_items': MAX_ITEMS,
    'prompt_template': PROMPT_TEMPLATE,
    'role_order': ROLE_ORDER,
    'prime_conditions': PRIME_CONDITIONS,
    'max_new_tokens': MAX_NEW_TOKENS,
    'seed': SEED,
    'output_root': OUTPUT_ROOT,
})

In [ ]:
%cd /content/Geometry-of-Syntax
!python scripts/19_run_colab_experiment_2.py \
  --model-name {MODEL_NAME} \
  --device cuda \
  --batch-size {BATCH_SIZE} \
  --max-items {MAX_ITEMS} \
  --prompt-template {PROMPT_TEMPLATE} \
  --role-order {ROLE_ORDER} \
  --which {WHICH} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --seed {SEED} \
  --output-root {OUTPUT_ROOT} \
  --prime-conditions {PRIME_CONDITIONS_ARG}

In [ ]:
%cd /content/Geometry-of-Syntax
!python scripts/20_report_choice_runs.py --root {OUTPUT_ROOT}

In [ ]:
from pathlib import Path
import pandas as pd

root = Path('/content/Geometry-of-Syntax') / OUTPUT_ROOT
folders = sorted(p for p in root.iterdir() if p.is_dir())

for folder in folders:
    print('\n===', folder.name, '===')
    summary_path = folder / 'summary.csv'
    if summary_path.exists():
        display(pd.read_csv(summary_path))
    gen_summary_path = folder / 'generation_summary.csv'
    if gen_summary_path.exists():
        display(pd.read_csv(gen_summary_path).head(20))

priming_path = root / 'priming_framed_results.csv'
if priming_path.exists():
    print('\n=== priming_framed_results ===')
    display(pd.read_csv(priming_path))

In [ ]:
from google.colab import files

%cd /content/Geometry-of-Syntax
!zip -r experiment_2_colab_results.zip {OUTPUT_ROOT}
files.download('experiment_2_colab_results.zip')